# 01 — Análise Exploratória e Entendimento dos Dados

**ORION AIOps** · Challenge FIAP x Locaweb 2026 · Turma 2TSCP

Antes de treinar qualquer modelo, minha regra foi não escrever uma linha de
código de machine learning sem entender profundamente os dados. Este notebook
é onde respondo três perguntas para mim mesmo antes de seguir em frente:

1. O que é, de fato, uma linha desta base?
2. O dado bate com o que o Dicionário de Dados descreve?
3. Onde está a dor real da operação — o problema que vale a pena resolver?

Os achados que documento aqui condicionam **todas** as decisões que tomo nos
notebooks seguintes. Se eu errar o diagnóstico agora, todo o resto desmorona.

In [ ]:
import sys, pathlib, warnings

# Descobre a pasta src/orion subindo a partir do diretório atual do kernel.
# Evita o erro "No module named 'orion'": o VS Code às vezes abre o notebook
# com o cwd na raiz do projeto, às vezes em notebooks/ — um caminho relativo
# fixo como '../src' só funciona no segundo caso.
_cwd = pathlib.Path.cwd()
for _base in [_cwd, *_cwd.parents]:
    _src = _base / 'src'
    if (_src / 'orion').is_dir():
        sys.path.insert(0, str(_src))
        break
else:
    raise FileNotFoundError(
        f"Não encontrei a pasta src/orion a partir de {_cwd}. "
        "Rode o notebook com o kernel na raiz do projeto (orion-aiops/) ou em notebooks/."
    )

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

## 1. Carga e grão da base

O primeiro passo que aprendi a sempre fazer é confirmar o *grão* da base:
o que representa, de fato, uma linha? Aqui, cada registro é um incidente
operacional registrado na plataforma de ITSM. Parece óbvio, mas só tenho
certeza disso depois de checar — e é essa checagem que evita erros de
agregação mais adiante.

In [ ]:
from orion.ingest import ingerir
from orion.transform import construir_silver

# Roda uma vez; nas execuções seguintes lê o Parquet já materializado
try:
    silver = pd.read_parquet('../data/silver/incidentes.parquet')
except FileNotFoundError:
    ingerir()
    silver = construir_silver()

print(f'{len(silver):,} incidentes | {silver.shape[1]} colunas')
print(f"Período: {silver['dt_abertura'].min():%d/%m/%Y} a {silver['dt_abertura'].max():%d/%m/%Y}")
silver.head(3)

## 2. Perfil de nulos

Depois de carregar os dados, o primeiro exame que faço é o perfil de nulos.
Foi aqui que veio meu primeiro sinal de alerta: mais de 60% dos registros
não têm taxonomia preenchida (`Categoria`, `Produto`, `Subcategoria`). Isso
me fez desconfiar que parte da base não é gerada por um humano abrindo
chamado — volto a esse ponto mais adiante.

In [ ]:
nulos = (silver.isna().mean() * 100).sort_values(ascending=False)
nulos = nulos[nulos > 0]

fig, ax = plt.subplots(figsize=(11, 6))
nulos.plot(kind='barh', ax=ax, color=np.where(nulos > 50, '#EF4444', '#38BDF8'))
ax.set_xlabel('% de valores nulos')
ax.set_title('Perfil de nulos — vermelho indica campo majoritariamente vazio')
plt.tight_layout(); plt.show()

nulos.round(1).to_frame('% nulo')

## 3. Composição da base

Segui a pista dos nulos e cheguei à minha primeira descoberta central: dois
terços da base são **ruído de monitoramento** — alarmes abertos automaticamente
que se autorresolvem sem nenhuma intervenção humana. Entendi que isso muda tudo:
esses registros não entram no KPI contratual e, portanto, não podem entrar na
modelagem de esforço operacional. Se eu tivesse modelado em cima da base crua,
estaria prevendo ruído de sistema, não trabalho real da equipe.

In [ ]:
composicao = pd.DataFrame({
    'Total da base':            [len(silver)],
    'Ruído de monitoramento':   [int(silver['is_ruido_monitoramento'].sum())],
    'Entram no KPI':            [int(silver['entrou_kpi'].sum())],
    'Violaram OLA':             [int(silver['ola_violado'].fillna(False).sum())],
}).T.rename(columns={0: 'incidentes'})
composicao['% da base'] = (composicao['incidentes'] / len(silver) * 100).round(1)
composicao

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, titulo in zip(
    axes,
    ['prioridade', 'status', 'origem_abertura'],
    ['Prioridade', 'Status', 'Origem da abertura'],
):
    vc = silver[col].value_counts()
    ax.barh(vc.index.astype(str), vc.values, color='#38BDF8')
    ax.set_title(titulo)
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 4. ACHADO CRÍTICO — quebra de regime em setembro/2025

Ao plotar a série de volume total, encontrei algo que quase me fez sair
modelando o dado errado: a série **não é estacionária**, ela muda de patamar
inteiro em setembro/2025. Se eu tivesse treinado um modelo com o histórico
completo sem investigar esse salto, teria produzido previsões inúteis — o
modelo aprenderia um padrão que não existe mais.

In [ ]:
mensal = silver.groupby(silver['dt_abertura'].dt.to_period('M')).size()
mensal.index = mensal.index.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(mensal.index, mensal.values, marker='o', color='#38BDF8', lw=2)
ax.axvline(pd.Timestamp('2025-09-01'), color='#EF4444', ls='--', lw=2)
ax.annotate('Quebra de regime\n(nova instrumentação)', xy=(pd.Timestamp('2025-09-01'), mensal.max()*0.7),
            xytext=(pd.Timestamp('2024-06-01'), mensal.max()*0.8), color='#EF4444',
            arrowprops=dict(arrowstyle='->', color='#EF4444'))
ax.set_title('Volume mensal total — o salto de set/2025 não é aumento de falhas')
ax.set_ylabel('incidentes/mês')
plt.tight_layout(); plt.show()

In [ ]:
# Isolando a causa: é ruído de monitoramento, não operação real
recorte = silver[silver['dt_abertura'] >= '2025-06-01']
causa = pd.crosstab(recorte['ano_mes'], recorte['is_ruido_monitoramento'])
causa.columns = ['Operação real', 'Ruído de monitoramento']
causa.plot(kind='bar', stacked=True, figsize=(12, 4.5), color=['#10B981', '#94A3B8'])
plt.title('A explosão é ruído de alarme, não degradação da infraestrutura')
plt.ylabel('incidentes'); plt.xlabel(''); plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

causa

In [ ]:
# Prova final: a série que entra no KPI permanece estável
kpi = silver[silver['entrou_kpi'] & silver['prioridade'].isin(['2 - Alta', '3 - Média'])]
serie_kpi = pd.crosstab(kpi['ano_mes'], kpi['prioridade']).tail(12)

serie_kpi.plot(figsize=(12, 4.5), marker='o', color=['#F59E0B', '#38BDF8'])
plt.title('Série de incidentes que entram no KPI — estável durante toda a quebra')
plt.ylabel('incidentes/mês'); plt.xlabel(''); plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

serie_kpi

## 5. ACHADO CRÍTICO — a regra de OLA do cliente diverge do dicionário

Enquanto validava o dado contra o Dicionário de Dados, notei uma discrepância
que quase passou despercebida. O dicionário diz que violação = `Duração >
limite da prioridade` (4h para P2, 12h para P3). Recalculei essa regra de
forma independente e comparei meu cálculo com o flag `KPI Violado?` que já
vem na base, para ver se batiam.

In [ ]:
k = silver[silver['entrou_kpi']].copy()
matriz = pd.crosstab(
    k['ola_violado_calc'].map({True: 'Cálculo: SIM', False: 'Cálculo: NÃO'}),
    k['ola_violado'].map({True: 'Flag: SIM', False: 'Flag: NÃO'}),
)
print(matriz)
print(f"\nDivergências: {int(k['ola_divergencia'].sum()):,} "
      f"({k['ola_divergencia'].mean():.1%} dos incidentes de KPI)")
print('\nPor prioridade:')
print(k[k['ola_divergencia']]['prioridade'].value_counts())

In [ ]:
# Hipótese: o relógio de OLA pausa. Evidência 1 — há violações ABAIXO do limite.
for prio, lim in [('2 - Alta', 4*3600), ('3 - Média', 12*3600)]:
    sub = k[(k['prioridade'] == prio) & (k['ola_violado'] == True)]
    print(f'{prio}: limite {lim/3600:.0f}h | duração mínima entre os violados: '
          f'{sub["duracao_s"].min()/3600:.1f}h')

# Evidência 2 — os que excedem mas não são violados concentram-se num status
excedem = k[(k['duracao_s'] > k['ola_limite_s']) & (k['ola_violado'] == False)]
print(f'\nExcedem o limite mas não violaram: {len(excedem):,}')
print(excedem['status'].value_counts())

**O que concluí:** deve existir uma regra de pausa de relógio (espera do
solicitante, janela de mudança ou horário comercial) que não está descrita
no dataset. Não consegui reproduzir essa regra só com as colunas disponíveis,
e decidi não tentar "adivinhar" a lógica exata — isso seria inventar dado.

**Decisão que tomei:** o alvo dos meus modelos passa a ser o flag `KPI
Violado?` do cliente, porque é a verdade contratual, não o meu recálculo.
Deixei a divergência exposta na coluna `ola_divergencia` — em vez de escondê-la,
transformo esse ponto em pergunta para a banca da Locaweb.

## 6. ACHADO CRÍTICO — as metas são degraus, não rampa

Esse foi o achado que mais mudou minha forma de pensar o projeto inteiro —
o insight que passei a usar para orientar toda a solução.

In [ ]:
from orion.config import META_OLA_QUEBRADOS, META_VOLUME_ANUAL, faixa_atingimento

linhas = []
for (ano, prio), g in k[k['prioridade'].isin(['2 - Alta', '3 - Média'])].groupby(['ano', 'prioridade']):
    vol, viol = len(g), int(g['ola_violado'].fillna(False).sum())
    linhas.append({
        'ano': ano, 'prioridade': prio, 'volume': vol, 'violações': viol,
        'taxa': f'{viol/vol:.2%}',
        'atingimento volume': f"{faixa_atingimento(META_VOLUME_ANUAL, prio, vol)}%",
        'atingimento OLA': f"{faixa_atingimento(META_OLA_QUEBRADOS, prio, viol)}%",
    })
pd.DataFrame(linhas)

In [ ]:
# Visualizando os degraus de P2 em 2025
faixas = META_OLA_QUEBRADOS['2 - Alta']
real_2025 = 42

fig, ax = plt.subplots(figsize=(12, 4))
for low, high, pct in faixas:
    largura = min(high, 60) - low
    cor = '#10B981' if pct >= 125 else '#38BDF8' if pct == 100 else '#F59E0B' if pct >= 50 else '#EF4444'
    ax.barh(0, largura, left=low, color=cor, edgecolor='white', height=0.5)
    ax.text(low + largura/2, 0, f'{pct}%', ha='center', va='center',
            color='white', fontweight='bold')
ax.axvline(real_2025, color='black', lw=3)
ax.text(real_2025, 0.4, f'Realizado 2025: {real_2025}', ha='center', fontweight='bold')
ax.set_xlim(0, 60); ax.set_ylim(-0.5, 0.7); ax.set_yticks([])
ax.set_xlabel('violações de OLA no ano')
ax.set_title('P2 — faixas de atingimento. Três violações a menos valiam 25 pontos percentuais.')
plt.tight_layout(); plt.show()

Com 42 violações, P2 caiu na faixa de **75%**. A faixa de 100% termina em 39.
Ou seja: **três incidentes** — só três — separam 75% de 100% de atingimento
contratual. Foi nesse momento que entendi por que uma taxa média de 0,81%
pode parecer excelente num dashboard e, ainda assim, esconder um problema
sério: o que importa aqui não é a taxa, é a **contagem absoluta contra o
degrau**. Decidi que era esse o número que o `AgentePerformance` deveria
monitorar, não a taxa.

## 7. Sazonalidade — o sinal mais forte da série

Com os achados críticos documentados, voltei a olhar a série do dia a dia
para entender qual padrão eu precisava capturar nos modelos de volume.

In [ ]:
kpi_2025 = k[(k['dt_abertura'] >= '2025-01-01') & k['prioridade'].isin(['2 - Alta', '3 - Média'])]
diario = kpi_2025.groupby([kpi_2025['data'], 'prioridade']).size().unstack(fill_value=0)
diario['dow'] = diario.index.dayofweek

dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
por_dow = diario.groupby('dow').mean()
por_dow.index = dias

por_dow.plot(kind='bar', figsize=(11, 4.5), color=['#F59E0B', '#38BDF8'])
plt.title('Média diária por dia da semana (2025) — domingo tem 1/4 do volume de terça')
plt.ylabel('incidentes/dia'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

por_dow.round(1)

In [ ]:
# Mapa de calor hora x dia da semana
heat = kpi_2025.pivot_table(index='hora', columns='dia_semana', values='incidente_id', aggfunc='count')
heat.columns = dias

plt.figure(figsize=(10, 7))
sns.heatmap(heat, cmap='YlOrRd', linewidths=0.4, cbar_kws={'label': 'incidentes'})
plt.title('Concentração por hora e dia da semana')
plt.ylabel('hora do dia'); plt.xlabel('')
plt.tight_layout(); plt.show()

## 8. Onde a dor se concentra

Por fim, quis descobrir em quais grupos, produtos e ativos a dor operacional
realmente se concentra, para saber onde recomendar atuação prioritária.

Uma coisa que uso bastante aqui é o **p90** (percentil 90) da duração. Não é
a duração média — é o valor abaixo do qual ficam 90% dos chamados. Uso isso
em vez da média porque a média se deixa enganar por poucos casos extremos
(um chamado que ficou aberto meses distorce a média inteira). O p90 responde
uma pergunta mais útil pra operação: "na grande maioria dos casos, em quanto
tempo isso se resolve?", ignorando os poucos casos fora da curva.

In [ ]:
def ranking(dim, top=8):
    r = (kpi_2025.groupby(dim, observed=True)
         .agg(volume=('incidente_id', 'count'),
              violacoes=('ola_violado', lambda s: int(s.fillna(False).sum())),
              duracao_p90_h=('duracao_h', lambda s: s.quantile(0.9)))
         .sort_values('violacoes', ascending=False).head(top))
    r['taxa_violacao'] = (r['violacoes'] / r['volume'] * 100).round(2)
    return r

for dim in ['grupo_designado', 'produto', 'categoria']:
    print(f'\n===== {dim} =====')
    print(ranking(dim).to_string())

In [ ]:
# Matriz de priorização: volume x taxa de violação
g = (kpi_2025.groupby('grupo_designado', observed=True)
     .agg(volume=('incidente_id', 'count'),
          violacoes=('ola_violado', lambda s: int(s.fillna(False).sum())),
          p90=('duracao_h', lambda s: s.quantile(0.9))))
g = g[g['volume'] >= 100]
g['taxa'] = g['violacoes'] / g['volume'] * 100

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(g['volume'], g['taxa'], s=g['violacoes']*12 + 40, alpha=0.65, color='#38BDF8')
for nome, r in g.iterrows():
    ax.annotate(nome, (r['volume'], r['taxa']), fontsize=9,
                xytext=(5, 5), textcoords='offset points')
ax.axhline(g['taxa'].median(), color='gray', ls='--', lw=1)
ax.axvline(g['volume'].median(), color='gray', ls='--', lw=1)
ax.set_xlabel('volume de incidentes KPI'); ax.set_ylabel('taxa de violação de OLA (%)')
ax.set_title('Quadrante superior direito = alto volume + alta taxa = prioridade de atuação')
plt.tight_layout(); plt.show()

In [ ]:
# Ativos reincidentes — candidatos a gestão de Problema (ITIL)
reinc = (kpi_2025.groupby('item_configuracao', observed=True)
         .agg(ocorrencias=('incidente_id', 'count'),
              violacoes=('ola_violado', lambda s: int(s.fillna(False).sum())),
              primeira=('dt_abertura', 'min'), ultima=('dt_abertura', 'max')))
reinc['meses'] = ((reinc['ultima'] - reinc['primeira']).dt.days / 30).clip(lower=1)
reinc['freq_mensal'] = (reinc['ocorrencias'] / reinc['meses']).round(1)
reinc.nlargest(10, 'ocorrencias')[['ocorrencias', 'violacoes', 'freq_mensal']]

## 9. Síntese dos achados

Fechando este notebook, organizei os seis achados que carrego comigo para o
resto do projeto — cada um virou uma decisão concreta de modelagem:

| # | Achado | Consequência no projeto |
|---|---|---|
| 1 | P2 fechou 2025 em 75% por 3 violações | Vira a métrica principal do BI e do `AgentePerformance` |
| 2 | Flag de OLA diverge da regra em 3.399 casos | Alvo = flag do cliente; divergência vira pergunta para a banca |
| 3 | Quebra de regime em set/2025 | `DATA_INICIO_REGIME = 2025-01-01` |
| 4 | 65,6% da base é ruído de monitoramento | Filtro `entrou_kpi` em toda a modelagem |
| 5 | Sazonalidade semanal forte | Features cíclicas + baseline `mesmo_dow_4s` |
| 6 | IC00349 gera 120 incidentes/mês | Recomendação de gestão de Problema |

Com esse diagnóstico em mãos, sigo para a etapa de engenharia de features:
`02_feature_engineering.ipynb`.